# PerturbAlign Quickstart Tutorial

This tutorial demonstrates how to use **PerturbAlign** to clean, standardize, and resolve multi-modal perturbation data (Perturb-seq / Sci-Plex / Mix-seq) with multi-dimensional biological and chemical knowledge profiling.

---
### Tutorial Outline
1. **Multi-Modal Perturbation Label Cleaning & Parsing** (Small molecules, genetic knockouts, microenvironmental stress)
2. **Single-Cell AnnData Standardization (UPDS v1.0 Specification)**
3. **Host Cell Line Context-Specific Causal Cascades & Pruning**
4. **Cross-Species Ortholog Mapping & Multi-Dimensional Knowledge Queries**
5. **Embedded Graph Database & Causal Subgraph Querying**


In [ ]:
# 1. Environment verification
import perturbalign
import pandas as pd
print("PerturbAlign version:", perturbalign.__version__)


## Step 1: Multi-Modal Perturbation Label Cleaning & Parsing
Single-cell screening datasets typically contain complex noise such as salt forms, duration/dose suffixes, vendor prefixes, and investigational drug codes.
PerturbAlign utilizes 22 built-in curated knowledge vocabularies to deliver robust, multi-modal entity alignment.


In [ ]:
raw_labels = [
    "Doxorubicin HCl 10nM 24h",    # Small molecule + salt form + concentration + duration
    "AZD-6244 1uM",               # Small molecule investigational code (Selumetinib)
    "sgTrp53 KO",                  # Genetic knockout (mouse Trp53)
    "Trametinib + Pictilisib",     # Combination therapy (Combo)
    "DMSO 0.1%",                   # Vehicle control (Control)
    "Hypoxia 1% O2 12h",           # Microenvironmental stress
]

# One-line multi-modal alignment
results = perturbalign.align(raw_labels)
for r in results:
    print(f"Raw: {r.raw_name:<28} -> Canonical: {r.parsed_name:<16} Modality: {r.modality:<14} Dose: {r.parsed_dose} {r.parsed_unit}")


## Step 2: Single-Cell AnnData Standardization (UPDS v1.0 Specification)
`perturbalign.standardize_anndata` standardizes single-cell AnnData objects according to the UPDS v1.0 specification:
- Concentrations are normalized to micromolar (`pert_dose_norm_um`)
- Durations are normalized to hours (`pert_duration_h`)
- Gene perturbations are mapped to human HGNC Symbols (`pert_human_ortholog`)
- Knowledge profiles and audit logs are recorded in `adata.uns['perturbalign']`


In [ ]:
# Mock an unstandardized single-cell perturbation dataset
import anndata as ad
import numpy as np

n_cells = 6
X = np.random.randn(n_cells, 20)
obs = pd.DataFrame({
    "perturbation": raw_labels,
    "cell_line": ["K562", "A549", "K562", "MCF7", "K562", "A549"],
})
adata = ad.AnnData(X=X, obs=obs)

# One-line standardization to UPDS format
adata = perturbalign.standardize_anndata(
    adata,
    obs_col="perturbation",
    cell_line_col="cell_line",
    online=False,
)

# Validate UPDS compliance and inspect standardized columns
report = perturbalign.validate_upds(adata)
print("UPDS Compliance:", report["is_valid"])
display_cols = ["pert_canonical_name", "pert_modality", "pert_dose_norm_um", "pert_duration_h", "pert_human_ortholog"]
adata.obs[display_cols]


## Step 3: Cell Line Context-Specific Causal Cascades & Pruning
Different cell lines (e.g. TP53-deficient K562 vs TP53-wildtype A549) exhibit distinct signaling downstream responses to identical perturbations.
PerturbAlign natively supports context-aware causal pruning based on host genomic alterations.


In [ ]:
# Scenario 1: TOP2A inhibition (e.g. Doxorubicin) in TP53-null K562 cells
k562_cascade = perturbalign.infer_causal_cascade(["TOP2A"], cell_line="K562")
print("=== K562 Context (TP53 null/mutant) ===")
print("Global downstream TFs:", k562_cascade["downstream_tfs"])
print("Active TFs after pruning:", k562_cascade["active_tfs"])      # TP53 is pruned
print("Pruned TFs & reasons:", k562_cascade["pruned_tfs"])
print("Cascade confidence score:", k562_cascade["cascade_confidence_score"]) # 0.65 (attenuated)

print("\n=== A549 Context (KRAS mutant, MEK hyperactive) ===")
a549_cascade = perturbalign.infer_causal_cascade(["MAP2K1", "MAP2K2"], cell_line="A549")
print("Active TFs:", a549_cascade["active_tfs"])
print("DoRothEA context-specific regulon (top 5):", a549_cascade["context_specific_regulon"][:5])
print("Cascade confidence score:", a549_cascade["cascade_confidence_score"]) # 0.97 (high confidence)


## Step 4: Cross-Species Orthologs & Multi-Dimensional Knowledge Queries
PerturbAlign provides direct access to curated databases including NCBI, STRING PPI, DepMap, and PubChem:


In [ ]:
# 1. Mouse query -> cross-species human ortholog and multi-omics profiling
gene_info = perturbalign.query_gene('Trp53', organism='mouse')
print('Mouse Trp53 human ortholog Symbol:', gene_info['ncbi']['symbol'])
if gene_info.get('esm2_embedding') is not None:
    print('ESM-2 embedding dimension:', len(gene_info['esm2_embedding']))
else:
    print('STRING PPI partners:', [item['interactor_symbol'] for item in gene_info['string_network']['interactors'][:3]])

# 2. Investigational pharma code resolution to canonical structure and targets
drug_info = perturbalign.query_compound('AZD-6244', online=False)
print('Canonical drug name:', drug_info['pubchem']['canonical_name'])
print('Canonical SMILES:', drug_info['pubchem']['canonical_smiles'])
print('Pharmacological targets:', drug_info['drugbank_annotations']['targets'])


## Step 5: Embedded Graph Database & Causal Network Querying
Persist heterogeneous biological knowledge graphs into embedded SQLite graph storage and extract multi-hop causal subgraphs:


In [ ]:
# Query knowledge graph neighborhood subgraph
kg = perturbalign.HeterogeneousKnowledgeGraph(load_curated=False)
subgraph = kg.get_subgraph('TP53', radius=1)
print('TP53 Neighborhood Subgraph node count:', subgraph.node_count)
print('TP53 Neighborhood Subgraph edge count:', subgraph.edge_count)
